In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_FILE = "../data/loan_applications.csv"
pd.set_option("display.width", 120)

In [2]:
## Loading dataset
df = pd.read_csv(DATA_FILE)

In [3]:
## Table Description
def table_desc(dataframe,columns = []):
    if len(columns) == 0:
        cols = dataframe.columns
    else:
        cols = columns
    col_description = []
    for col in cols:
        col_description.append([
            col,
            dataframe[col].dtypes,
            dataframe[col].notna().sum(),
            dataframe[col].isna().sum(),
            round((dataframe[col].isna().sum()/dataframe.shape[0])*100,2),
            dataframe[col].nunique(),
            dataframe[col].duplicated().sum(),
            dataframe[col].max() if dataframe[col].dtype in ['int64','float64','Float64'] else 'N/A',
            dataframe[col].min() if dataframe[col].dtype in ['int64','float64','Float64'] else 'N/A',
            dataframe[col].mean() if dataframe[col].dtype in ['int64','float64','Float64'] else 'N/A',
            dataframe[col].std() if dataframe[col].dtype in ['int64','float64','Float64'] else 'N/A',
            dataframe[col].quantile(0.25) if dataframe[col].dtype in ['int64','float64','Float64'] else 'N/A',
            dataframe[col].quantile(0.5) if dataframe[col].dtype in ['int64','float64','Float64'] else 'N/A',
            dataframe[col].quantile(0.75) if dataframe[col].dtype in ['int64','float64','Float64'] else 'N/A'
            ])

    df_desc = pd.DataFrame(col_description,columns=['Column','dtype','nb_not_null','nb_null','%_null','nb_unique','nb_duplicates','max','min','mean','std','25%','50%','75%'])
    # print(df_desc)
    return df_desc

In [4]:
print("Dataset Shape: ", df.shape)
print("Number of duplicated rows:")
print(df.duplicated().sum())
print("\nTable description of the dataset:")
describe_table = table_desc(df)
display(describe_table)


Dataset Shape:  (25375, 13)
Number of duplicated rows:
375

Table description of the dataset:


,Column,dtype,nb_not_null,nb_null,%_null,nb_unique,nb_duplicates,max,min,mean,std,25%,50%,75%
0,application_id,object,25375,0,0.00,25000,375,N/A,N/A,N/A,N/A,N/A,N/A,N/A
1,applicant_age,int64,25375,0,0.00,49,25326,69,21,44.955153,14.065724,33.0,45.0,57.0
2,employment_status,object,25375,0,0.00,14,25361,N/A,N/A,N/A,N/A,N/A,N/A,N/A
3,annual_income,float64,23835,1540,6.07,1025,24349,126200.0,9000.0,55007.593875,20312.378468,42000.0,56100.0,69100.0
4,credit_score,float64,24369,1006,3.96,390,24984,850.0,436.0,670.329148,65.132888,626.0,671.0,714.0
5,loan_amount,float64,25375,0,0.00,503,24872,60000.0,1100.0,13751.672906,7349.251248,8700.0,12100.0,16900.0
6,loan_term_months,int64,25375,0,0.00,5,25370,60,12,35.35401,14.016535,24.0,36.0,48.0
7,interest_rate,float64,25375,0,0.00,890,24485,13.8,3.5,7.266248,1.666391,6.13,7.25,8.4
8,debt_to_income_pct,object,25375,0,0.00,1295,24080,N/A,N/A,N/A,N/A,N/A,N/A,N/A
9,home_ownership,object,25375,0,0.00,3,25372,N/A,N/A,N/A,N/A,N/A,N/A,N/A


In [5]:
## removing not necessary columns, dropping duplicates, normalizing columns,and fixing data types

## dropping duplicates and unnecessary columns
df_copy = df.drop_duplicates().copy()
df_removed_columns = df_copy.drop(columns=["application_id", "collections_referral"])

# normalizing column names
categorical_cols = [
    "employment_status",
    "home_ownership"
]

for col in categorical_cols:
    df_removed_columns[col] = (
        df_removed_columns[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

for col in categorical_cols:
    df_removed_columns[col] = (
        df_removed_columns[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# normalizing column employment_status
df_removed_columns["employment_status"] = (
    df_removed_columns["employment_status"]
    .str.replace(" ", "-", regex=False)
)

# converting debt-to-income ratio to numeric
df_removed_columns["debt_to_income_pct"] = pd.to_numeric(
    df_removed_columns["debt_to_income_pct"]
    .astype("string")
    .str.strip()
    .str.replace("%", "", regex=False),
    errors="coerce"
)

describe_table = table_desc(df_removed_columns)
print("Dataset Shape after removing duplicates and columns with all null values: ", df_removed_columns.shape)
print("Dataset description after removing duplicates and columns with all null values: ")
display(describe_table.T)

Dataset Shape after removing duplicates and columns with all null values:  (25000, 11)
Dataset description after removing duplicates and columns with all null values: 


,0,1,2,3,4,5,6,7,8,9,10
Column,applicant_age,employment_status,annual_income,credit_score,loan_amount,loan_term_months,interest_rate,debt_to_income_pct,home_ownership,previous_defaults,defaulted
dtype,int64,string[python],float64,float64,float64,int64,float64,Float64,string[python],int64,int64
nb_not_null,25000,25000,23484,24015,25000,25000,25000,25000,25000,25000,25000
nb_null,0,0,1516,985,0,0,0,0,0,0,0
%_null,0.0,0.0,6.06,3.94,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nb_unique,49,4,1025,390,503,5,890,712,3,4,2
nb_duplicates,24951,24996,23974,24609,24497,24995,24110,24288,24997,24996,24998
max,69,N/A,126200.0,850.0,60000.0,60,13.8,75.0,N/A,3,1
min,21,N/A,9000.0,436.0,1100.0,12,3.5,2.0,N/A,0,0
mean,44.94532,N/A,55004.935275,670.300229,13746.884,35.3616,7.268332,34.1287,N/A,0.3116,0.20124


In [6]:
# Checking cleaned dataset
print("Dataset shape:", df_removed_columns.shape)
print("\nMissing values:")
print(df_removed_columns.isna().sum())
print("\nData types:")
print(df_removed_columns.dtypes)
print("\nTarget distribution:")
print(df_removed_columns["defaulted"].value_counts(normalize=True))

Dataset shape: (25000, 11)

Missing values:
applicant_age            0
employment_status        0
annual_income         1516
credit_score           985
loan_amount              0
loan_term_months         0
interest_rate            0
debt_to_income_pct       0
home_ownership           0
previous_defaults        0
defaulted                0
dtype: int64

Data types:
applicant_age                  int64
employment_status     string[python]
annual_income                float64
credit_score                 float64
loan_amount                  float64
loan_term_months               int64
interest_rate                float64
debt_to_income_pct           Float64
home_ownership        string[python]
previous_defaults              int64
defaulted                      int64
dtype: object

Target distribution:
defaulted
0    0.79876
1    0.20124
Name: proportion, dtype: float64


In [7]:
# Saving cleaned dataset
df_removed_columns.to_csv("../data/loan_applications_cleaned_dataset.csv", index=False)